In [1]:
import torch
import torch.nn as nn
from torchvision import datasets,transforms
from torch.utils.data import DataLoader,random_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
from torch.cuda import manual_seed
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/Breast_Cancer/dataset_cancer_v1/classificacao_binaria/40X'

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

full_datasets = datasets.ImageFolder(
    path,
    transform=transform
)

train_size = int(0.8*len(full_datasets))
val_size = int(0.1*len(full_datasets))
test_size = len(full_datasets) - train_size - val_size

train_dataset,val_dataset,test_dataset=random_split(
    full_datasets,
    [train_size,val_size,test_size],
    generator = torch.Generator().manual_seed(42)
)





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Data Loader

train_loader = DataLoader(
    train_dataset,
    batch_size= 32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [5]:
from torchvision.models import vit_b_16
model = vit_b_16(weights="IMAGENET1K_V1")

model.heads.head = nn.Linear(model.heads.head.in_features,2)

model = model.to(device)

In [6]:
import torch.optim

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)

In [7]:
epochs = 10
best_loss = float("inf")

for epoch in range(epochs):
    running_loss = 0
    corrected = 0
    totaL = 0

    model.train()
    for images, lables in train_loader:
        images = images.to(device)
        lables = lables.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, lables)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    training_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicated = torch.max(outputs, 1)
            corrected += (predicated == labels).sum().item()
            totaL += labels.size(0)

    valdation_loss = val_loss / len(val_loader)
    val_accy = corrected / totaL * 100

    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {training_loss:.4f}, "
          f"Validation Loss: {valdation_loss:.4f}, Validation Acc: {val_accy:.2f}%")

    if valdation_loss < best_loss:
        best_loss = valdation_loss
        torch.save(model.state_dict(), "best_model.pth")

Epoch 1/10, Training Loss: 0.6787, Validation Loss: 0.5967, Validation Acc: 69.35%
Epoch 2/10, Training Loss: 0.5600, Validation Loss: 0.4620, Validation Acc: 77.89%
Epoch 3/10, Training Loss: 0.4330, Validation Loss: 0.7489, Validation Acc: 63.82%
Epoch 4/10, Training Loss: 0.2544, Validation Loss: 0.1264, Validation Acc: 94.97%
Epoch 5/10, Training Loss: 0.1471, Validation Loss: 0.0725, Validation Acc: 98.49%
Epoch 6/10, Training Loss: 0.0916, Validation Loss: 0.1358, Validation Acc: 96.98%
Epoch 7/10, Training Loss: 0.1424, Validation Loss: 0.2404, Validation Acc: 89.95%
Epoch 8/10, Training Loss: 0.0616, Validation Loss: 0.1436, Validation Acc: 95.48%
Epoch 9/10, Training Loss: 0.0457, Validation Loss: 0.2075, Validation Acc: 96.98%
Epoch 10/10, Training Loss: 0.0307, Validation Loss: 0.2094, Validation Acc: 96.98%


In [8]:
model.load_state_dict(torch.load("best_model.pth"))

<All keys matched successfully>

In [10]:
#Accurcy

model.eval()
correct =0
total = 0

with torch.no_grad():

  for images,labels in test_loader:

    images = images.to(device)
    labels = labels.to(device)

    output = model(images)

    _, predicated = torch.max(output,1)

    total+= labels.size(0)

    correct += (predicated == labels).sum().item()

test_accy = 100*correct / total

print(f"Test Accuracy: {test_accy:.4f}")


Test Accuracy: 95.5000
